# (a) Tạo dataset YOLO-seg bằng pseudo-label tự động

Chạy trên Kaggle với Internet bật. Notebook này chỉ tạo `/kaggle/working/dataset`; tải folder đó thành Kaggle Dataset rồi add vào notebook train riêng.

Input cần add: `pthttm-teachers`, `water-dataset`, `flood-yolo-seg-dataset`. Notebook chỉ đọc ảnh từ dataset cũ và **bỏ qua tuyệt đối mọi `labels/`, `.txt`, `data.yaml`** trong đó. WaterDataset có mask thật được dùng làm flood label; một phần group WaterDataset giữ riêng làm test. Các ảnh còn lại phải qua cả ba teacher và bước đồng thuận nước.

Road lấy từ SegFormer ADE20K nhưng chỉ giữ vùng đủ tin cậy (ngưỡng ở cell cấu hình road), luôn trừ flood và không chồng lấn flood. Ảnh có road không đáng tin bị bỏ hoặc giữ không có nhãn road, tuỳ `ROAD_UNRELIABLE_POLICY`.

Class order cố định: `0 flood`, `1 road`, `2 car_L0`, `3 car_L1`, `4 car_L2`, `5 car_L3`, `6 car_L4`.


In [ ]:
# Cell 1: khám phá Kaggle Input, tìm weights và cặp ảnh/mask mà không giả định mount path.
from pathlib import Path
import re, os, hashlib, random, json, csv, shutil, math, collections, time

# Trial mode: maximum images sampled from each source. Set None to process all images.
LIMIT = 50
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise FileNotFoundError('Không thấy /kaggle/input. Hãy chạy notebook trong Kaggle và Add ba input dataset.')

def show_tree(root, maxdepth=4):
    print(f'\nCây thư mục {root} (maxdepth={maxdepth}):')
    for p in sorted(root.rglob('*')):
        try: depth=len(p.relative_to(root).parts)
        except ValueError: continue
        if depth > maxdepth: continue
        print('  '*depth + ('[D] ' if p.is_dir() else '[F] ') + p.name)
show_tree(INPUT_ROOT, 4)

def norm(s): return re.sub(r'[^a-z0-9]', '', str(s).lower())
def dirs_named(root, names):
    wanted={norm(x) for x in names}
    return [p for p in root.rglob('*') if p.is_dir() and norm(p.name) in wanted]
def files_named(root, filename):
    return [p for p in root.rglob(filename) if p.is_file()]

water_weights=files_named(INPUT_ROOT,'link_efficientb4_model.pth')
car_weights=files_named(INPUT_ROOT,'best_car_YOLOv11.pt')
if not water_weights: raise FileNotFoundError('Không tìm thấy **/link_efficientb4_model.pth. Kiểm tra pthttm-teachers đã được Add vào notebook.')
if not car_weights: raise FileNotFoundError('Không tìm thấy **/best_car_YOLOv11.pt. Kiểm tra pthttm-teachers đã được Add vào notebook.')
WATER_WEIGHTS=water_weights[0]; CAR_LEVEL_WEIGHTS=car_weights[0]
print('[FOUND] WaterNet weight candidates:',*[str(p) for p in water_weights],sep='\n  ')
print('[FOUND] Car-level weight candidates:',*[str(p) for p in car_weights],sep='\n  ')

# Locate WaterDataset (either dataset mount root or nested archive extraction).
water_roots=dirs_named(INPUT_ROOT,['WaterDataset'])
if not water_roots: water_roots=dirs_named(INPUT_ROOT,['water-dataset'])
if not water_roots: raise FileNotFoundError('Không tìm thấy thư mục WaterDataset/water-dataset trong /kaggle/input.')
WATER_ROOT=water_roots[-1] if any(norm(p.name)=='waterdataset' for p in water_roots) else water_roots[0]
old_roots=[p for p in INPUT_ROOT.rglob('*') if p.is_dir() and norm(p.name)=='floodyolosegdataset']
if not old_roots: raise FileNotFoundError('Không tìm thấy thư mục flood_yolo_seg_dataset (hoặc tên tương đương) trong /kaggle/input.')
OLD_ROOT=old_roots[0]
roadway_roots=[p for p in INPUT_ROOT.rglob('*') if p.is_dir() and norm(p.name)=='roadwayfloodingimagedataset']
ROADWAY_ROOT=roadway_roots[0] if roadway_roots else None
print('\nFound WaterNet:',WATER_WEIGHTS)
print('Found car-level model:',CAR_LEVEL_WEIGHTS)
print('WaterDataset root:',WATER_ROOT)
print('Old image dataset root (labels will be ignored):',OLD_ROOT)
print('Roadway Flooding dataset root:',ROADWAY_ROOT if ROADWAY_ROOT else 'NOT ATTACHED (skip; add Kaggle input to include)')

# Discover likely image/mask locations immediately and fail early on a wrong input.
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
MASK_EXTS=IMAGE_EXTS|{'.tif','.tiff','.npy'}
MASK_HINTS=('mask','groundtruth','annotation','label','gt')
water_image_dirs=[p for p in WATER_ROOT.rglob('*') if p.is_dir() and 'image' in norm(p.name) and not any(t in norm(p.name) for t in MASK_HINTS) and any(q.is_file() and q.suffix.lower() in IMAGE_EXTS for q in p.rglob('*'))]
water_mask_dirs=[p for p in WATER_ROOT.rglob('*') if p.is_dir() and any(t in norm(p.name) for t in MASK_HINTS)]
water_mask_files=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and (any(t in norm(x) for x in p.parts for t in MASK_HINTS) or any(p.stem.lower().endswith(s) for s in ('_mask','-mask','_label','-label','_gt','-gt','_water','-water')))]
if not water_image_dirs: raise FileNotFoundError(f'WaterDataset image directory not found under {WATER_ROOT}; expected train_images/JPEGImages or an equivalent images folder.')
if not water_mask_dirs and not water_mask_files: raise FileNotFoundError(f'WaterDataset mask directory/files not found under {WATER_ROOT}; inspect printed tree and add the correct dataset version.')
print('WaterDataset image dirs:',*[str(p) for p in water_image_dirs[:20]],sep='\n  ')
print('WaterDataset mask dirs:',*[str(p) for p in water_mask_dirs[:20]],sep='\n  ')
print('WaterDataset mask-like files:',len(water_mask_files))
print('Candidate WaterDataset image roots:',len(water_image_dirs),'| mask roots:',len(water_mask_dirs))

## 0. Cấu hình road mask

Road là pseudo-label duy nhất lấy từ SegFormer ADE20K, không có nhãn tay. Các ngưỡng dưới đây quyết định pixel/vùng/ảnh nào được gán road. Giá trị mặc định chưa được hiệu chỉnh; hãy chạy `LIMIT = 50`, xem `preview/`, `preview_road_rejected/`, `road_debug.csv` rồi chỉnh.

In [ ]:
# Cell cấu hình road: mọi ngưỡng dùng để lấy road từ SegFormer ADE20K.
# Giá trị mặc định là điểm xuất phát hợp lý, CHƯA hiệu chỉnh trên dữ liệu thật.
# Chạy LIMIT=50, xem preview + road_debug.csv rồi chỉnh ở đây.

# Lớp ADE20K được coi là road. Chỉ 'road': không gộp sidewalk/path/floor/earth để không biến
# vỉa hè, sân, nền đất thành road.
ROAD_CLASS_NAMES=('road',)

# (1) Ngưỡng pixel. Pixel là road khi argmax = road VÀ P(road) >= ROAD_PIXEL_CONF.
#     Softmax có 150 lớp nên argmax có thể thắng với P(road) chỉ ~0.2; 0.5 nghĩa là road chiếm đa số xác suất.
ROAD_PIXEL_CONF=0.50
#     Pixel có P(water+sea+river+lake) > ngưỡng này là mặt đường/nước lẫn lộn (thường là nước ngập
#     mà flood mask bỏ sót) -> không gán road.
ROAD_MAX_WATER_PROB=0.20

# (2) Làm sạch hình thái học. Kernel = ROAD_MORPH_FRAC * cạnh ngắn của ảnh (tối thiểu 3 px):
#     opening xoá đốm nhiễu/viền mảnh, closing nối khe hở nhỏ.
ROAD_MORPH_FRAC=0.01
# Lỗ kín nhỏ hơn tỉ lệ ảnh này bên trong một vùng (road hoặc flood) được lấp như nhiễu.
REGION_FILL_HOLE_FRAC=0.002
# Khoảng đệm (px) giữa road và flood: road = road_pred - dilate(flood). Đệm phải lớn hơn
# 2*REGION_POLY_EPS_PX + 1 để polygon đã đơn giản hoá của road và flood không chạm nhau.
ROAD_FLOOD_GAP_PX=4
# Sai số đơn giản hoá polygon (px) cho flood/road (trước đây là 0.2% chu vi, có thể tới hàng chục px).
REGION_POLY_EPS_PX=1.0

# (3) Lọc từng vùng road liên thông.
ROAD_MIN_REGION_FRAC=0.01      # vùng < 1% ảnh là mảnh vụn -> bỏ (lý do: small)
ROAD_REGION_MEAN_CONF=0.70     # P(road) trung bình của vùng < 0.70 -> bỏ (lý do: low_conf)

# (4) Kiểm tra cấp ảnh trên road còn giữ lại. Vi phạm bất kỳ điều kiện nào -> road của ảnh 'unreliable'.
ROAD_MAX_TOTAL_FRAC=0.85       # road > 85% ảnh là bất thường (cận cảnh mặt đường hoặc nhầm lớp)
ROAD_MAX_REGIONS=6             # quá nhiều vùng road rời nhau -> mask vụn
ROAD_IMAGE_MEAN_CONF=0.75      # P(road) trung bình theo pixel của road giữ lại
ROAD_MAX_UNCERTAIN_FRAC=0.10   # pixel argmax=road, không phải flood, nhưng bị loại > 10% ảnh
                               # -> nhiều mặt đường thật có thể sẽ không có nhãn

# (5) Xử lý ảnh có road 'unreliable'.
#   'drop_image'        : bỏ ảnh khỏi train/val (mặc định). YOLO-seg không có vùng ignore, nên giữ ảnh
#                         mà bỏ nhãn road sẽ dạy model rằng mặt đường nhìn thấy được là background.
#   'keep_without_road' : giữ ảnh (không mất flood/xe) nhưng không có polygon road (thêm false negative road).
# Ảnh test WaterDataset (flood GT) không bao giờ bị bỏ: nếu road unreliable thì giữ ảnh, không có nhãn road.
ROAD_UNRELIABLE_POLICY='drop_image'

# (6) Kiểm tra cuối trên nhãn: rasterize polygon (giống Ultralytics). Phần polygon road trùng
#     flood (polygon + mask) phải <= tỉ lệ này của polygon road; vượt thì bỏ polygon road đó.
ROAD_FLOOD_MAX_OVERLAP_FRAC=0.0

# Số preview của ảnh bị bỏ vì road, lưu riêng để review.
ROAD_REJECT_PREVIEWS=50

assert ROAD_UNRELIABLE_POLICY in ('drop_image','keep_without_road')
assert ROAD_FLOOD_GAP_PX>2*REGION_POLY_EPS_PX+1, 'ROAD_FLOOD_GAP_PX quá nhỏ so với REGION_POLY_EPS_PX'
print('[ROAD CONFIG]',{k:v for k,v in globals().items() if k.startswith(('ROAD_','REGION_'))})


## 1. Cài thư viện teacher

In [ ]:
%pip -q install segmentation-models-pytorch==0.2.0 transformers ultralytics


## 2. Tìm ảnh WaterDataset có ground-truth mask và ảnh nguồn cũ

Mask được nhận diện theo tên thư mục (`mask`, `masks`, `groundtruth`, `annotations`, …) và/hoặc hậu tố filename. Nếu không tìm ra cặp ảnh/mask thì notebook dừng để tránh vô tình coi mask là ảnh.

In [ ]:
import numpy as np
import cv2
from PIL import Image
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
MASK_EXTS=IMAGE_EXTS|{'.tif','.tiff','.npy'}
MASK_DIR_TOKENS=('mask','masks','groundtruth','groundtruths','annotation','annotations','label','labels','gt')
MASK_SUFFIXES=('_mask','-mask','_label','-label','_gt','-gt','_water','-water')
GENERIC_DIRS={'images','image','imgs','img','frames','frame','train','val','valid','validation','test','trainimages','valimages','validimages','testimages','waterdataset','waterdatasetzip'}

def is_mask_dir(p): return any(t in norm(x) for x in p.parts for t in MASK_DIR_TOKENS)
def stem_key(p):
    s=p.stem.lower()
    for suffix in MASK_SUFFIXES:
        if s.endswith(suffix): s=s[:-len(suffix)]; break
    return s

def read_binary_mask(p):
    if p.suffix.lower()=='.npy': a=np.load(p)
    else: a=np.array(Image.open(p))
    if a.ndim==3: a=a[...,0]
    return a>0

water_all=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not is_mask_dir(p) and not any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
water_masks=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and is_mask_dir(p)]
# Also include sibling mask-named files (e.g. frame_mask.png beside frame.jpg).
water_masks += [p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
mask_by_stem=collections.defaultdict(list)
for p in water_masks: mask_by_stem[stem_key(p)].append(p)
water_pairs=[]
for im in water_all:
    candidates=mask_by_stem.get(im.stem.lower(),[])
    if not candidates: candidates=mask_by_stem.get(stem_key(im),[])
    if candidates:
        # Prefer masks nested in a mask/GT folder over incidental similarly named files.
        candidates=sorted(set(candidates),key=lambda p:(not is_mask_dir(p),len(p.parts)))
        water_pairs.append((im,candidates[0]))
if not water_pairs:
    mask_dirs=[str(p) for p in WATER_ROOT.rglob('*') if p.is_dir() and is_mask_dir(p)]
    raise FileNotFoundError('WaterDataset image/mask pairs not found. Mask-like folders: '+repr(mask_dirs[:30])+'; inspect the printed tree and update the mask-pair matcher.')

# Old dataset: deliberately skip every labels directory and every annotation/config file.
old_images=[]
for p in OLD_ROOT.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS: continue
    rel_parts=[norm(x) for x in p.relative_to(OLD_ROOT).parts]
    if any(x in {'label','labels','annotation','annotations'} for x in rel_parts[:-1]): continue
    if any(x.endswith('yaml') or x.endswith('yml') for x in rel_parts): continue
    old_images.append(p)
if not old_images: raise FileNotFoundError('Không tìm thấy ảnh trong flood-yolo-seg-dataset sau khi bỏ qua labels/.')
print(f'WaterDataset: {len(water_all)} ảnh, {len(water_masks)} mask candidates, {len(water_pairs)} cặp ảnh/mask thật')
print(f'Old dataset: {len(old_images)} ảnh được lấy; nội dung labels/ và data.yaml không được mở.')

def scan_roadway(root):
    imgs=[p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not is_mask_dir(p) and not any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
    masks=[p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and (is_mask_dir(p) or any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES))]
    lookup=collections.defaultdict(list)
    for p in masks: lookup[stem_key(p)].append(p)
    pairs=[]
    for im in imgs:
        ms=lookup.get(im.stem.lower(),[]) or lookup.get(stem_key(im),[])
        if ms: pairs.append((im,sorted(set(ms),key=lambda p:(not is_mask_dir(p),len(p.parts)))[0]))
    return imgs,masks,pairs

if ROADWAY_ROOT is not None:
    roadway_all,roadway_masks,roadway_pairs=scan_roadway(ROADWAY_ROOT)
    if not roadway_all: raise FileNotFoundError(f'Không tìm thấy ảnh trong {ROADWAY_ROOT}.')
    roadway_gt={im:mask for im,mask in roadway_pairs}
    print(f'Roadway Flooding: {len(roadway_all)} ảnh, {len(roadway_masks)} mask candidates, {len(roadway_pairs)} cặp khớp.')
    if roadway_pairs: print('Roadway mask files match image stems: dùng làm flood GT.')
    else: print('Không có mask khớp ảnh trong Roadway Flooding: dùng ảnh này cho teacher pseudo-label.')
else:
    roadway_all=[]; roadway_masks=[]; roadway_pairs=[]; roadway_gt={}
    print('[INPUT MISSING] Roadway Flooding chưa có trong cây /kaggle/input; source sẽ được bỏ qua lần chạy này.')

## 3. Tạo group theo video/nguồn và giữ test ground-truth riêng

Mọi frame trong cùng thư mục video nằm trong cùng group. Dataset cũ được chia group theo thư mục con/video nếu có; nếu ảnh được làm phẳng trong một thư mục, group được suy ra từ prefix filename. Test gồm 15% group WaterDataset có mask thật; toàn bộ group đó bị loại khỏi train/val để tránh leakage.

In [ ]:
SEED=42; TEST_GROUP_FRACTION=0.15; VAL_GROUP_FRACTION=0.15
random.seed(SEED)
def source_group(path, root, source_tag):
    rel=path.relative_to(root)
    parts=[x for x in rel.parts[:-1] if norm(x) not in GENERIC_DIRS and not is_mask_dir(Path(x))]
    if parts:
        # deepest meaningful directory is the video/source folder
        return source_tag+'/'+('/'.join(parts))
    # Without reliable per-video names, keep a flat source together as one group.
    # This prevents adjacent video frames from leaking across train and validation.
    return source_tag+'/unstructured-source'

water_groups=collections.defaultdict(list)
for im,mask in water_pairs: water_groups[source_group(im,WATER_ROOT,'water')].append((im,mask))
if len(water_groups)<2:
    raise ValueError(f'WaterDataset có {len(water_groups)} group mask thật; cần ít nhất 2 group để giữ test riêng theo video/nguồn.')
all_water_group_names=sorted(water_groups)
rng=random.Random(SEED); rng.shuffle(all_water_group_names)
n_test=max(1,round(len(all_water_group_names)*TEST_GROUP_FRACTION))
test_groups=set(all_water_group_names[:n_test])
print(f'{len(water_groups)} GT groups; test giữ {len(test_groups)} groups, {sum(len(water_groups[g]) for g in test_groups)} ảnh.')

# Collect all candidate images. GT images use true masks; other WaterDataset images and
# all old images are sent through all three teachers.
records=[]
for group,items in water_groups.items():
    split='test' if group in test_groups else None
    for im,mask in items: records.append({'image':im,'gt_mask':mask,'source':'water','group':group,'split':split})
paired_images={im for im,_ in water_pairs}
for im in water_all:
    if im in paired_images: continue
    if 'test_images_from_other_papers' in [x.lower() for x in im.parts]: continue
    group=source_group(im,WATER_ROOT,'water')
    if group in test_groups: continue
    records.append({'image':im,'gt_mask':None,'source':'water','group':group,'split':None})
for im in old_images:
    group=source_group(im,OLD_ROOT,'old')
    records.append({'image':im,'gt_mask':None,'source':'old','group':group,'split':None})
for im in roadway_all:
    group=source_group(im,ROADWAY_ROOT,'roadway')
    records.append({'image':im,'gt_mask':roadway_gt.get(im),'source':'roadway','group':group,'split':None})

full_source_counts=dict(collections.Counter(r['source'] for r in records))
def choose_group_balanced(items,cap,rng):
    if cap is None or len(items)<=cap: return list(items)
    buckets=collections.defaultdict(list)
    for item in items: buckets[item['group']].append(item)
    keys=list(buckets); rng.shuffle(keys)
    for key in keys: rng.shuffle(buckets[key])
    chosen=[]
    while len(chosen)<cap and keys:
        next_keys=[]
        for key in keys:
            if buckets[key] and len(chosen)<cap:
                chosen.append(buckets[key].pop())
            if buckets[key]: next_keys.append(key)
        keys=next_keys
    return chosen

if LIMIT is not None:
    limited=[]
    for source in sorted({r['source'] for r in records}):
        source_items=[r for r in records if r['source']==source]
        rng=random.Random(SEED+sum(map(ord,source)))
        # Reserve a representative test sample from the GT WaterDataset before sampling the rest.
        test_items=[r for r in source_items if r['split']=='test']
        other_items=[r for r in source_items if r['split']!='test']
        reserve=min(len(test_items),max(1,round(LIMIT*TEST_GROUP_FRACTION))) if test_items else 0
        selected=choose_group_balanced(test_items,reserve,rng)
        selected+=choose_group_balanced(other_items,max(0,LIMIT-len(selected)),rng)
        limited.extend(selected)
        print(f'[LIMIT] {source}: {len(source_items)} -> {len(selected)} images (limit {LIMIT})')
    records=limited
print('Candidates after LIMIT:',len(records),'| true-mask:',sum(r['gt_mask'] is not None for r in records),'| no-mask:',sum(r['gt_mask'] is None for r in records))

## 4. Clone V-FloodNet và load đúng WaterNet với `strict=True`

Repo lưu `link_efficientb4_model.pth` như model PyTorch. Theo `train_image_seg.py` của repo, kiến trúc là `smp.Linknet(encoder_name="efficientnet-b4", encoder_depth=5, encoder_weights="imagenet", in_channels=3, classes=1, activation="sigmoid")`. Load model pickle của repo để lấy state, dựng đúng định nghĩa LinkNet từ repo, rồi `load_state_dict(..., strict=True)`. Bất kỳ lỗi nào dừng notebook.

In [ ]:
import subprocess, sys
from pathlib import Path
VF_REPO=Path('/kaggle/working/V-FloodNet')
if not VF_REPO.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/xmlyqing00/V-FloodNet.git',str(VF_REPO)],check=True)
sys.path.insert(0,str(VF_REPO))
import torch
import segmentation_models_pytorch as smp
from test_image_seg import predict_pil

try:
    load_t0=time.perf_counter(); print(f'[LOAD] WaterNet checkpoint: {WATER_WEIGHTS}')
    serialized=torch.load(WATER_WEIGHTS,map_location='cpu',weights_only=False)
    if isinstance(serialized,torch.nn.Module): state=serialized.state_dict()
    elif isinstance(serialized,dict):
        state=serialized.get('state_dict',serialized.get('weights',serialized))
    else: raise TypeError(f'Unexpected WaterNet checkpoint type: {type(serialized)}')
    water_model=smp.Linknet(
        encoder_name='efficientnet-b4', encoder_depth=5, encoder_weights=None,
        in_channels=3, classes=1, activation='sigmoid')
    water_model.load_state_dict(state,strict=True)
    water_model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[LOAD OK] WaterNet architecture=strict Linknet EfficientNet-B4, {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] WaterNet: {type(e).__name__}: {e}')
    raise RuntimeError('WaterNet không load được với kiến trúc chính thức Linknet/EfficientNet-B4 và strict=True; dừng để không tạo nhãn sai.') from e
print('WaterNet loaded strict=True:',type(water_model).__name__)

## 5. Load SegFormer ADE20K và hai teacher ô tô

In [ ]:
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
from ultralytics import YOLO
from torchvision.ops import box_iou
import numpy as np, cv2, re, collections

DEVICE=0 if torch.cuda.is_available() else 'cpu'
seg_name='nvidia/segformer-b0-finetuned-ade-512-512'
load_t0=time.perf_counter(); print(f'[LOAD] SegFormer ADE20K: {seg_name}')
try:
    seg_processor=AutoImageProcessor.from_pretrained(seg_name)
    seg_model=SegformerForSemanticSegmentation.from_pretrained(seg_name).eval()
    if torch.cuda.is_available(): seg_model=seg_model.cuda()
    print(f'[LOAD OK] SegFormer ADE20K in {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] SegFormer ADE20K: {type(e).__name__}: {e}')
    raise
label_by_id={int(i):str(name).strip().lower() for i,name in seg_model.config.id2label.items()}
road_ids={i for i,n in label_by_id.items() if n in {x.lower() for x in ROAD_CLASS_NAMES}}
water_names={'water','sea','river','lake'}
water_class_ids={i for i,n in label_by_id.items() if n in water_names}
if len(road_ids)!=len(set(ROAD_CLASS_NAMES)): raise ValueError(f'ROAD_CLASS_NAMES={ROAD_CLASS_NAMES} không khớp hết config.id2label; tìm thấy {road_ids}: {label_by_id}')
if not water_class_ids: raise ValueError(f'SegFormer ADE20K không có water/sea/river/lake trong config.id2label: {label_by_id}')
print('ADE20K road ids:',{i:label_by_id[i] for i in road_ids},'water ids:',{i:label_by_id[i] for i in water_class_ids})
load_t0=time.perf_counter(); print(f'[LOAD] Car-level YOLO: {CAR_LEVEL_WEIGHTS}')
try:
    car_level_model=YOLO(str(CAR_LEVEL_WEIGHTS))
    print(f'[LOAD OK] Car-level YOLO in {time.perf_counter()-load_t0:.1f}s; classes={car_level_model.names}')
except Exception as e:
    print(f'[LOAD ERROR] Car-level YOLO: {type(e).__name__}: {e}'); raise
load_t0=time.perf_counter(); print('[LOAD] YOLO11x-seg COCO weights (download on first run if not cached)')
try:
    car_segmenter=YOLO('yolo11x-seg.pt')
    print(f'[LOAD OK] YOLO11x-seg in {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] YOLO11x-seg: {type(e).__name__}: {e}'); raise

## 6. Teacher inference, đồng thuận flood, road tin cậy, xe theo IoU, và polygon labels

Flood masks từ WaterNet và SegFormer được so bằng IoU trên ảnh không có nhãn thật. Khi cả hai dự đoán rỗng, ảnh được xem là negative candidate. Polygon flood bị loại nếu diện tích dưới 0.5% ảnh.

**Road** (ngưỡng ở cell cấu hình road):
1. *Pixel*: argmax SegFormer = road **và** softmax P(road) ≥ `ROAD_PIXEL_CONF` **và** P(nước) ≤ `ROAD_MAX_WATER_PROB`. Trước đây chỉ dùng argmax, nên pixel có P(road)≈0.2 vẫn thành road.
2. *Làm sạch*: opening/closing, lấp lỗ nhỏ.
3. **`road = road_prediction − flood_mask`**, trừ thêm khoảng đệm `ROAD_FLOOD_GAP_PX`.
4. *Vùng*: bỏ vùng liên thông quá nhỏ (`small`) hoặc có P(road) trung bình thấp (`low_conf`).
5. *Ảnh*: road giữ lại quá lớn, quá vụn, confidence thấp, hoặc còn nhiều pixel argmax=road bị loại (`uncertain_road`) → road **unreliable**. Khi đó ảnh bị bỏ (`drop_image`) hoặc giữ nhưng không có nhãn road; ảnh test luôn được giữ.
6. *Polygon*: lỗ trong vùng flood/road được giữ bằng khe cắt. Code cũ dùng `RETR_EXTERNAL`, nên một vùng flood nằm giữa mặt đường bị polygon road phủ kín. Mỗi polygon road được rasterize và kiểm tra lại, không được trùng flood.

Mọi ảnh qua bước road đều ghi vào `road_debug.csv` với status, confidence, diện tích, số vùng và lý do.

**Hạn chế**: road chỉ đến từ SegFormer-B0 ADE20K, model train trên ảnh cảnh chung chứ không phải đường ngập; confidence softmax cũng chưa được calibrate. Ngưỡng chỉ giúp loại bớt nhãn road sai, không biến road thành ground truth. Mặt đường bị nước che được gán flood, không phải road.

In [ ]:
OUT=Path('/kaggle/working/dataset')
for split in ('train','val','test'):
    (OUT/'images'/split).mkdir(parents=True,exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True,exist_ok=True)
PREVIEW=Path('/kaggle/working/preview'); PREVIEW.mkdir(parents=True,exist_ok=True)
CLASS_NAMES=['flood','road','car_L0','car_L1','car_L2','car_L3','car_L4']
MIN_REGION_FRAC=0.005; AGREEMENT_IOU=0.5; CAR_CONF=0.5; CAR_IOU=0.5

def waternet_mask(pil):
    # Uses the official repo's preprocessing/inference/postprocessing function.
    pred=predict_pil(water_model,pil,model_dims=(416,416),device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    return np.array(pred)>0

def segformer_maps(pil):
    # Returns full-resolution argmax water/road masks (as before) plus P(road) and P(water-like).
    inputs=seg_processor(images=pil.convert('RGB'),return_tensors='pt')
    dev=next(seg_model.parameters()).device
    inputs={k:v.to(dev) for k,v in inputs.items()}
    with torch.inference_mode():
        logits=seg_model(**inputs).logits
        # Softmax at the model's native resolution, then upsample only the two summed channels (saves memory).
        probs=logits.float().softmax(1)
        pr=torch.stack([probs[:,sorted(road_ids)].sum(1),probs[:,sorted(water_class_ids)].sum(1)],1)
        pr=torch.nn.functional.interpolate(pr,size=(pil.height,pil.width),mode='bilinear',align_corners=False)[0].clamp_(0,1).cpu().numpy()
        logits=torch.nn.functional.interpolate(logits,size=(pil.height,pil.width),mode='bilinear',align_corners=False)
        ids=logits.argmax(1)[0].cpu().numpy()
    water=np.isin(ids,list(water_class_ids)); road=np.isin(ids,list(road_ids))
    return water,road,pr[0],pr[1]

def contours_polygons(mask):
    # Used for car instances (unchanged behaviour).
    mask=np.asarray(mask,dtype=bool)
    h,w=mask.shape; min_area=MIN_REGION_FRAC*h*w
    contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    rows=[]; accepted=[]
    for contour in contours:
        area=cv2.contourArea(contour)
        if area<min_area: continue
        poly=cv2.approxPolyDP(contour,0.002*cv2.arcLength(contour,True),True).reshape(-1,2)
        if len(poly)<3: continue
        rows.append(' '.join(f'{x/w:.6f} {y/h:.6f}' for x,y in poly))
        accepted.append((poly,area))
    return rows,accepted

# ---------- Region (flood/road) mask helpers ----------
def odd_ellipse(size_px):
    k=max(3,int(round(size_px))); k+=(k%2==0)
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))

def _background_components(pad):
    # 4-connected background components of a zero-padded mask; the padding is one component (outside).
    n,lab,stats,_=cv2.connectedComponentsWithStats((~pad).astype(np.uint8),connectivity=4)
    return n,lab,stats,int(lab[0,0])

def fill_small_holes(mask,max_area):
    # Fill enclosed background holes whose area <= max_area (speckle inside a region).
    pad=np.pad(np.asarray(mask,bool),1,constant_values=False)
    n,lab,stats,outside=_background_components(pad)
    small=[i for i in range(1,n) if i!=outside and stats[i,cv2.CC_STAT_AREA]<=max_area]
    if small: pad=pad|np.isin(lab,small)
    return pad[1:-1,1:-1]

def bridge_holes(mask,slit=2,max_iter=50):
    # YOLO-seg polygons cannot hold holes; RETR_EXTERNAL would silently fill them (a flood area
    # inside a road region became road). Cut a thin vertical slit from each hole to the outside so
    # the outer contour walks around the hole and the filled polygon excludes it.
    pad=np.pad(np.asarray(mask,bool),1,constant_values=False).copy()
    for _ in range(max_iter):
        n,lab,stats,outside=_background_components(pad)
        holes=[i for i in range(1,n) if i!=outside]
        if not holes: break
        for i in holes:
            y0=int(stats[i,cv2.CC_STAT_TOP]); x0=int(np.flatnonzero(lab[y0]==i)[0]); y=y0-1
            while pad[y,x0]:  # padding row 0 is background, so the loop always stops
                pad[y,x0:x0+slit]=False; y-=1
    return pad[1:-1,1:-1]

def mask_to_polygons(mask,min_frac,eps_px=None):
    # Hole-aware polygons for region classes; returns integer pixel polygons (N,2).
    eps_px=REGION_POLY_EPS_PX if eps_px is None else eps_px
    mask=np.asarray(mask,bool); h,w=mask.shape; min_area=min_frac*h*w
    if not mask.any(): return []
    pad=np.pad(bridge_holes(mask),1,constant_values=False).astype(np.uint8)
    contours,_=cv2.findContours(pad,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
    polys=[]
    for contour in contours:
        if cv2.contourArea(contour)<min_area: continue
        poly=cv2.approxPolyDP(contour,eps_px,True).reshape(-1,2)-1
        poly=np.clip(poly,0,[w-1,h-1])
        if len(poly)>=3: polys.append(poly)
    return polys

def poly_to_row(poly,w,h): return ' '.join(f'{x/w:.6f} {y/h:.6f}' for x,y in poly)

def rasterize(polys,h,w):
    # Same rasterizer Ultralytics uses for polygon labels (cv2.fillPoly, one polygon at a time).
    out=np.zeros((h,w),bool)
    for poly in polys:
        m=np.zeros((h,w),np.uint8); cv2.fillPoly(m,[np.asarray(poly,np.int32).reshape(-1,1,2)],1); out|=m.astype(bool)
    return out

def outline_polys(mask,eps_px=2.0):
    # Light-weight outlines for preview/debug only.
    contours,_=cv2.findContours(np.asarray(mask,np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.approxPolyDP(c,eps_px,True).reshape(-1,2) for c in contours if len(c)>=3]

def analyze_road(road_argmax,road_prob,water_prob,flood,flood_poly_mask):
    """road = road_prediction - flood_mask, with pixel/region/image-level reliability checks.
    Returns (road_polys, debug). debug['status'] is 'ok', 'no_road' or 'unreliable'."""
    h,w=road_argmax.shape; hw=float(h*w)
    dbg={'raw_frac':float(road_argmax.mean())}
    water_ambig=road_argmax&(water_prob>ROAD_MAX_WATER_PROB)
    pred=road_argmax&(road_prob>=ROAD_PIXEL_CONF)&~water_ambig
    dbg['water_ambiguous_frac']=float(water_ambig.mean())
    dbg['pixel_conf_frac']=float(pred.mean())
    k=odd_ellipse(ROAD_MORPH_FRAC*min(h,w))
    pred=cv2.morphologyEx(pred.astype(np.uint8),cv2.MORPH_OPEN,k)
    pred=cv2.morphologyEx(pred,cv2.MORPH_CLOSE,k).astype(bool)
    pred=fill_small_holes(pred,REGION_FILL_HOLE_FRAC*hw)
    # road = road_prediction - flood_mask (plus a small buffer so simplified polygons cannot touch).
    flood_ref=flood|flood_poly_mask
    flood_buffer=cv2.dilate(flood_ref.astype(np.uint8),odd_ellipse(2*ROAD_FLOOD_GAP_PX+1)).astype(bool)
    road=pred&~flood_buffer
    assert not (road&flood).any(), 'road mask overlaps flood mask'
    # Region filtering.
    n,lab,stats,_=cv2.connectedComponentsWithStats(road.astype(np.uint8),connectivity=8)
    conf_sum=np.bincount(lab.ravel(),weights=road_prob.ravel(),minlength=n)
    kept_ids=[]; region_reasons=collections.Counter(); regions=[]
    for i in range(1,n):
        area=int(stats[i,cv2.CC_STAT_AREA]); mean_conf=float(conf_sum[i]/area)
        reason='small' if area<ROAD_MIN_REGION_FRAC*hw else ('low_conf' if mean_conf<ROAD_REGION_MEAN_CONF else None)
        regions.append((round(area/hw,4),round(mean_conf,3),reason or 'kept'))
        if reason: region_reasons[reason]+=1
        else: kept_ids.append(i)
    regions.sort(key=lambda r:-r[0])
    kept=np.isin(lab,kept_ids) if kept_ids else np.zeros_like(road)
    rejected=road&~kept
    dbg.update(n_regions=n-1,n_kept=len(kept_ids),n_rejected_small=region_reasons['small'],
               n_rejected_low_conf=region_reasons['low_conf'],regions=regions,
               kept_frac=float(kept.mean()),kept_conf=float(road_prob[kept].mean()) if kept.any() else 0.0,
               uncertain_frac=float((road_argmax&~flood_buffer&~kept).mean()))
    # Image-level plausibility.
    reasons=[]
    if dbg['kept_frac']>ROAD_MAX_TOTAL_FRAC: reasons.append('area_too_large')
    if dbg['n_kept']>ROAD_MAX_REGIONS: reasons.append('too_many_regions')
    if kept.any() and dbg['kept_conf']<ROAD_IMAGE_MEAN_CONF: reasons.append('low_image_conf')
    if dbg['uncertain_frac']>ROAD_MAX_UNCERTAIN_FRAC: reasons.append('uncertain_road')
    dbg['status']='unreliable' if reasons else ('ok' if kept.any() else 'no_road')
    dbg['reasons']=reasons
    # Polygons + label-level overlap check against flood (polygons and mask), as the trainer will rasterize them.
    polys=[]; dropped_overlap=0; max_overlap=0
    if dbg['status']=='ok':
        for poly in mask_to_polygons(kept,ROAD_MIN_REGION_FRAC):
            r=rasterize([poly],h,w); ov=int((r&flood_ref).sum()); max_overlap=max(max_overlap,ov)
            if ov>ROAD_FLOOD_MAX_OVERLAP_FRAC*r.sum(): dropped_overlap+=1; continue
            polys.append(poly)
        if not polys: dbg['status']='no_road'
    dbg.update(n_polys=len(polys),n_polys_dropped_overlap=dropped_overlap,poly_flood_overlap_px_max=max_overlap)
    # Debug artefacts for previews, kept small (PNG bytes) so LIMIT=None does not exhaust RAM.
    sw=160; sh=max(1,round(h*sw/w))
    dbg['prob_png']=cv2.imencode('.png',(cv2.resize(road_prob,(sw,sh),interpolation=cv2.INTER_AREA)*255).astype(np.uint8))[1].tobytes()
    dbg['rejected_outlines']=outline_polys(rejected)
    dbg['kept_outlines']=outline_polys(kept) if dbg['status']=='unreliable' else []
    return polys,dbg

def level_index(name):
    m=re.search(r'(?:level|l)[ _-]?([0-4])\s*$',str(name).lower())
    return int(m.group(1)) if m else None

def make_car_rows(image_path,w,h):
    level=car_level_model.predict(str(image_path),conf=CAR_CONF,device=DEVICE,verbose=False)[0]
    seg=car_segmenter.predict(str(image_path),classes=[2],conf=0.25,device=DEVICE,retina_masks=True,verbose=False)[0]
    if level.boxes is None or seg.boxes is None or seg.masks is None: return []
    lboxes=level.boxes.xyxy.cpu().numpy(); lcls=level.boxes.cls.cpu().numpy(); lconf=level.boxes.conf.cpu().numpy()
    sboxes=seg.boxes.xyxy.cpu().numpy(); smasks=seg.masks.data.cpu().numpy()
    rows=[]; used=set()
    for cls,box,conf in zip(lcls,lboxes,lconf):
        if conf<CAR_CONF: continue
        lv=level_index(level.names[int(cls)])
        if lv is None or len(sboxes)==0: continue
        overlaps=box_iou(torch.as_tensor(box).float().view(1,4),torch.as_tensor(sboxes).float())[0].numpy()
        order=np.argsort(overlaps)[::-1]
        match=next((int(i) for i in order if overlaps[i]>=CAR_IOU and int(i) not in used),None)
        if match is None: continue
        used.add(match)
        cm=cv2.resize((smasks[match]>0.5).astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
        # Clip segmentation mask to the classifier's box.
        x1,y1,x2,y2=box; dx=.03*(x2-x1); dy=.03*(y2-y1)
        roi=np.zeros((h,w),bool); roi[max(0,int(y1-dy)):min(h,int(y2+dy)),max(0,int(x1-dx)):min(w,int(x2+dx))]=True
        cm &= roi
        polys,_=contours_polygons(cm)
        rows.extend(f'{2+lv} {poly}' for poly in polys)
    return rows

def group_hash(p): return hashlib.sha1(str(p).encode()).hexdigest()[:12]
accepted=[]; dropped=collections.Counter(); instance_counts=collections.Counter(); pending_negative=[]
image_durations=[]; durations_by_source=collections.defaultdict(list)
road_log=[]        # one row per image that reached road analysis (kept or dropped)
road_rejected=[]   # images dropped because road was unreliable (for preview_road_rejected)
for idx,rec in enumerate(records):
    image_t0=time.perf_counter()
    src=rec['image']
    try:
        pil=Image.open(src).convert('RGB'); w,h=pil.size
        if rec['gt_mask'] is not None:
            flood=read_binary_mask(rec['gt_mask'])
            flood=cv2.resize(flood.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
            seg_water,road_raw,road_prob,water_prob=segformer_maps(pil) # still run all teachers, retaining road/car labels
            reason=None
        else:
            flood=waternet_mask(pil)
            seg_water,road_raw,road_prob,water_prob=segformer_maps(pil)
            a=int(flood.sum()); b=int(seg_water.sum())
            if a==0 and b==0:
                reason='negative'
            else:
                iou=float(np.logical_and(flood,seg_water).sum()/max(1,np.logical_or(flood,seg_water).sum()))
                reason='water_teacher_iou_below_0.5' if iou<AGREEMENT_IOU else None
        if reason and reason!='negative':
            dropped[reason]+=1
            elapsed=time.perf_counter()-image_t0; image_durations.append(elapsed); durations_by_source[rec['source']].append(elapsed)
            continue
        # Force same-size masks if source model preprocessing has rounded a dimension.
        flood=cv2.resize(flood.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
        flood=fill_small_holes(flood,REGION_FILL_HOLE_FRAC*w*h)
        road_raw=cv2.resize(road_raw.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
        rows=[]
        flood_polys=mask_to_polygons(flood,MIN_REGION_FRAC)
        if flood.any() and not flood_polys:
            dropped['flood_regions_below_0.5_percent']+=1
            elapsed=time.perf_counter()-image_t0; image_durations.append(elapsed); durations_by_source[rec['source']].append(elapsed)
            continue
        road_polys,road_dbg=analyze_road(road_raw,road_prob,water_prob,flood,rasterize(flood_polys,h,w))
        is_test=rec['split']=='test'
        road_dbg['action']='kept'
        if road_dbg['status']=='unreliable':
            if ROAD_UNRELIABLE_POLICY=='drop_image' and not is_test:
                road_dbg['action']='image_dropped'
                dropped['road_unreliable']+=1
                road_log.append({'record':rec,'road':road_dbg,'uid':None,'split':None})
                road_rejected.append({'record':rec,'road':road_dbg,'flood_polys':flood_polys})
                elapsed=time.perf_counter()-image_t0; image_durations.append(elapsed); durations_by_source[rec['source']].append(elapsed)
                continue
            road_dbg['action']='kept_without_road'
        rows += [f'0 {poly_to_row(p,w,h)}' for p in flood_polys]
        rows += [f'1 {poly_to_row(p,w,h)}' for p in road_polys]
        rows += make_car_rows(src,w,h)
        is_negative=(not flood.any())
        if rec['split']=='test': split='test'
        else: split=None
        uid=hashlib.sha1((str(src)+str(rec['gt_mask'])).encode()).hexdigest()[:16]
        accepted.append({'record':rec,'rows':rows,'uid':uid,'negative':is_negative,'split':split,'size':(w,h),'road':road_dbg})
        road_log.append(accepted[-1])
        if is_negative: pending_negative.append(accepted[-1])
        for line in rows: instance_counts[int(line.split(' ',1)[0])]+=1
    except Exception as e:
        dropped['teacher_or_decode_error']+=1
        print(f'[DROP] {src}: {type(e).__name__}: {e}')
    elapsed=time.perf_counter()-image_t0; image_durations.append(elapsed); durations_by_source[rec['source']].append(elapsed)
    if (idx+1)%10==0: print(f'[PROGRESS] {idx+1}/{len(records)} images; kept={len(accepted)}; road_dropped={dropped["road_unreliable"]}; avg={np.mean(image_durations):.2f}s/image')
print('Candidate labels generated:',len(accepted),'drop reasons:',dict(dropped))
print('Road status (ảnh đã qua phân tích road):',dict(collections.Counter((x['road']['status'],x['road']['action']) for x in road_log)))
for source in sorted(full_source_counts):
    vals=durations_by_source.get(source,[])
    avg=float(np.mean(vals)) if vals else 0.0
    print(f'[TIME] {source}: {avg:.2f}s/image average; estimated full source {full_source_counts[source]*avg/60:.1f} min for {full_source_counts[source]} images')
print(f'[TIME] Overall average {np.mean(image_durations):.2f}s/image across {len(image_durations)} sampled images' if image_durations else '[TIME] No images processed')


## 7. Group split 85/15, giới hạn negative tối đa 15%, và ghi YOLO dataset

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
# Split only groups outside held-out WaterDataset test groups. No frame-level split.
trainval=[x for x in accepted if x['split'] is None]
groups=sorted({x['record']['group'] for x in trainval})
if len(groups)<2: raise ValueError(f'Chỉ còn {len(groups)} source/video group cho train/val; cần ít nhất 2.')
tr_groups_idx,va_groups_idx=next(GroupShuffleSplit(n_splits=1,test_size=VAL_GROUP_FRACTION,random_state=SEED).split(groups,groups=groups))
train_groups={groups[i] for i in tr_groups_idx}; val_groups={groups[i] for i in va_groups_idx}
for x in trainval: x['split']='train' if x['record']['group'] in train_groups else 'val'
# Keep no-water samples at <=15% of complete retained dataset.
positive_count=sum(not x['negative'] for x in accepted)
max_neg=math.floor(positive_count*0.15/0.85)
negative_order=list(pending_negative); random.Random(SEED).shuffle(negative_order)
keep_negative_ids={x['uid'] for x in negative_order[:max_neg]}
filtered=[]
for x in accepted:
    if x['negative'] and x['uid'] not in keep_negative_ids:
        dropped['negative_cap_15_percent']+=1
        x['road']['action']='image_dropped_negative_cap'
    else: filtered.append(x)
accepted=filtered

for x in accepted:
    split=x['split']
    src=x['record']['image']; dst=OUT/'images'/split/(x['uid']+'.jpg')
    Image.open(src).convert('RGB').save(dst,quality=95)
    (OUT/'labels'/split/(x['uid']+'.txt')).write_text(('\n'.join(x['rows'])+'\n') if x['rows'] else '')

data_yaml=OUT/'data.yaml'
import yaml
with data_yaml.open('w') as f:
    yaml.safe_dump({'path':str(OUT.resolve()),'train':'images/train','val':'images/val','test':'images/test','names':{i:n for i,n in enumerate(CLASS_NAMES)}},f,sort_keys=False,allow_unicode=True)
print('Split counts:',dict(collections.Counter(x['split'] for x in accepted)))
print('Kept:',len(accepted),'Dropped by reason:',dict(dropped))

## 8. Preview 50 overlay ảnh ngẫu nhiên, debug road và thống kê

- `/kaggle/working/preview`: nhãn đã ghi (flood đỏ, road cyan, xe theo màu level); viền xám = vùng road bị lọc bỏ; góc phải trên = heatmap P(road); dòng chữ dưới = status/confidence/diện tích/số vùng/lý do.
- `/kaggle/working/preview_road_rejected`: ảnh bị bỏ vì road unreliable, viền cyan mảnh = road lẽ ra được gán.
- `/kaggle/working/road_debug.csv`: thống kê road từng ảnh; `auto_label_manifest.csv` thêm các cột road.
- Cell dừng với lỗi nếu nhãn road đã ghi chồng lấn flood.

In [ ]:
# Render polygons from the written YOLO labels with class colors, plus road debug:
# P(road) heatmap inset, grey outlines = road regions rejected by the filters, text = road stats.
COLORS=[(0,0,255),(255,160,0),(0,220,0),(0,180,255),(255,0,180),(180,0,255),(255,255,0)]
REJECT_COLOR=(160,160,160)

def road_text(d):
    reg=[f"{a*100:.1f}%/p{c:.2f}/{r}" for a,c,r in d['regions'][:4]]
    lines=[f"road {d['status']} ({d['action']}) conf={d['kept_conf']:.2f} area={d['kept_frac']*100:.1f}% raw={d['raw_frac']*100:.1f}%",
           f"regions keep={d['n_kept']} small={d['n_rejected_small']} lowconf={d['n_rejected_low_conf']} polys={d['n_polys']} uncertain={d['uncertain_frac']*100:.1f}% waterambig={d['water_ambiguous_frac']*100:.1f}%"]
    if d['reasons']: lines.append('image reasons: '+','.join(d['reasons']))
    if d['n_polys_dropped_overlap']: lines.append(f"polys dropped (flood overlap): {d['n_polys_dropped_overlap']}")
    if reg: lines.append('regions(area/meanP/decision): '+' '.join(reg)+(' ...' if len(d['regions'])>4 else ''))
    return lines

def draw_road_debug(canvas,d,title):
    h,w=canvas.shape[:2]
    for poly in d['rejected_outlines']:
        cv2.polylines(canvas,[np.asarray(poly,np.int32).reshape(-1,1,2)],True,REJECT_COLOR,1)
    for poly in d['kept_outlines']:  # only for unreliable images: road that would have been labelled
        cv2.polylines(canvas,[np.asarray(poly,np.int32).reshape(-1,1,2)],True,COLORS[1],1)
    # P(road) heatmap inset (top-right).
    heat=cv2.imdecode(np.frombuffer(d['prob_png'],np.uint8),cv2.IMREAD_GRAYSCALE)
    iw=max(80,w//4); ih=max(1,round(heat.shape[0]*iw/heat.shape[1]))
    if ih<h and iw<w:
        inset=cv2.applyColorMap(cv2.resize(heat,(iw,ih)),cv2.COLORMAP_JET)
        canvas[0:ih,w-iw:w]=inset
        cv2.putText(canvas,'P(road)',(w-iw+4,ih-6),cv2.FONT_HERSHEY_SIMPLEX,.45,(255,255,255),1)
    lines=[title]+road_text(d)
    scale=max(.4,min(.6,w/1600)); lh=int(22*scale/.5)
    cv2.rectangle(canvas,(0,h-lh*len(lines)-8),(w,h),(0,0,0),-1)
    for i,t in enumerate(lines):
        cv2.putText(canvas,t,(8,h-lh*(len(lines)-i-1)-10),cv2.FONT_HERSHEY_SIMPLEX,scale,(255,255,255),1,cv2.LINE_AA)
    return canvas

rng=random.Random(SEED); sample=rng.sample(accepted,min(50,len(accepted)))
for x in sample:
    split=x['split']; ip=OUT/'images'/split/(x['uid']+'.jpg')
    canvas=cv2.imread(str(ip)); overlay=canvas.copy(); h,w=canvas.shape[:2]
    label_path=OUT/'labels'/split/(x['uid']+'.txt')
    for line in label_path.read_text().splitlines():
        vals=line.split(); cls=int(vals[0]); coords=np.array([float(v) for v in vals[1:]],dtype=np.float32).reshape(-1,2)
        pts=np.round(coords*np.array([w,h])).astype(np.int32).reshape(-1,1,2)
        cv2.fillPoly(overlay,[pts],COLORS[cls])
        cv2.polylines(canvas,[pts],True,COLORS[cls],2)
    canvas=cv2.addWeighted(overlay,0.30,canvas,0.70,0)
    canvas=draw_road_debug(canvas,x['road'],f"{split}: {x['uid']} [{x['record']['source']}]")
    cv2.imwrite(str(PREVIEW/f"{x['uid']}.jpg"),canvas)

# Images dropped because road was unreliable: shown with flood labels (red) and the road that was refused.
PREVIEW_ROAD_REJECTED=Path('/kaggle/working/preview_road_rejected'); PREVIEW_ROAD_REJECTED.mkdir(parents=True,exist_ok=True)
for x in random.Random(SEED).sample(road_rejected,min(ROAD_REJECT_PREVIEWS,len(road_rejected))):
    canvas=cv2.cvtColor(np.array(Image.open(x['record']['image']).convert('RGB')),cv2.COLOR_RGB2BGR); overlay=canvas.copy()
    for poly in x['flood_polys']: cv2.fillPoly(overlay,[np.asarray(poly,np.int32).reshape(-1,1,2)],COLORS[0])
    canvas=cv2.addWeighted(overlay,0.30,canvas,0.70,0)
    name=hashlib.sha1(str(x['record']['image']).encode()).hexdigest()[:16]
    cv2.imwrite(str(PREVIEW_ROAD_REJECTED/f'{name}.jpg'),draw_road_debug(canvas,x['road'],f"DROPPED road unreliable [{x['record']['source']}]"))

# Safety net: re-read every written label and verify road polygons do not overlap flood polygons.
overlap_bad=[]
for x in accepted:
    w,h=x['size']; polys={0:[],1:[]}
    for line in (OUT/'labels'/x['split']/(x['uid']+'.txt')).read_text().splitlines():
        vals=line.split(); cls=int(vals[0])
        if cls in polys: polys[cls].append(np.round(np.array(vals[1:],np.float32).reshape(-1,2)*[w,h]))
    if polys[0] and polys[1]:
        ov=int((rasterize(polys[0],h,w)&rasterize(polys[1],h,w)).sum())
        if ov: overlap_bad.append((x['uid'],ov))
print(f'[CHECK] road/flood overlap in written labels: {len(overlap_bad)} ảnh có pixel chồng lấn',overlap_bad[:10])

counts=collections.Counter()
by_split=collections.Counter()
for x in accepted:
    by_split[x['split']]+=1
    for line in x['rows']:
        counts[CLASS_NAMES[int(line.split(' ',1)[0])]]+=1
print('Ảnh giữ:',len(accepted),'| ảnh bỏ:',sum(dropped.values()))
print('Bỏ theo lý do:',dict(dropped))
print('Ảnh theo split:',dict(by_split))
print('Instance mỗi class:',{name:counts.get(name,0) for name in CLASS_NAMES})
print('Preview:',len(list(PREVIEW.glob('*.jpg'))),'ảnh tại',PREVIEW,'| road bị loại:',len(list(PREVIEW_ROAD_REJECTED.glob('*.jpg'))),'ảnh tại',PREVIEW_ROAD_REJECTED)

# Road statistics over every image that reached road analysis.
print('\n=== Road statistics ===')
print('Status/action:',dict(collections.Counter((x['road']['status'],x['road']['action']) for x in road_log)))
for source in sorted({x['record']['source'] for x in road_log}):
    c=collections.Counter(x['road']['status'] for x in road_log if x['record']['source']==source)
    print(f'  {source}: {dict(c)}')
print('Lý do road unreliable (cấp ảnh):',dict(collections.Counter(r for x in road_log for r in x['road']['reasons'])))
print('Vùng road bị loại: small=',sum(x['road']['n_rejected_small'] for x in road_log),
      '| low_conf=',sum(x['road']['n_rejected_low_conf'] for x in road_log),
      '| polygon bỏ do chồng flood=',sum(x['road']['n_polys_dropped_overlap'] for x in road_log))
ok=[x['road'] for x in road_log if x['road']['status']=='ok']
if ok:
    for key in ('kept_conf','kept_frac','n_kept','uncertain_frac'):
        v=np.array([d[key] for d in ok],float)
        print(f'  road ok {key}: p10={np.percentile(v,10):.3f} median={np.median(v):.3f} p90={np.percentile(v,90):.3f}')
test_unrel=[x for x in road_log if x['split']=='test' and x['road']['status']=='unreliable']
if test_unrel: print(f'[WARN] {len(test_unrel)} ảnh test có road unreliable: giữ ảnh vì flood GT nhưng KHÔNG có nhãn road; road trong test vẫn là pseudo-label, không phải GT.')

ROAD_COLS=['status','action','reasons','kept_conf','kept_frac','raw_frac','pixel_conf_frac','water_ambiguous_frac','uncertain_frac',
           'n_regions','n_kept','n_rejected_small','n_rejected_low_conf','n_polys','n_polys_dropped_overlap','poly_flood_overlap_px_max','regions']
def road_values(d): return [';'.join(d[k]) if k=='reasons' else (json.dumps(d[k]) if k=='regions' else (round(d[k],4) if isinstance(d[k],float) else d[k])) for k in ROAD_COLS]
with (Path('/kaggle/working')/'road_debug.csv').open('w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['uid','image','source','group','split']+['road_'+k for k in ROAD_COLS])
    for x in road_log: wr.writerow([x['uid'],x['record']['image'],x['record']['source'],x['record']['group'],x['split']]+road_values(x['road']))
print('Road debug CSV:',Path('/kaggle/working')/'road_debug.csv')
# Traceability: all kept records and source path/group.
with (Path('/kaggle/working')/'auto_label_manifest.csv').open('w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['uid','image','source','group','split','negative','instances','road_status','road_action','road_conf','road_area_frac','road_regions','road_reasons'])
    for x in accepted:
        d=x['road']; wr.writerow([x['uid'],x['record']['image'],x['record']['source'],x['record']['group'],x['split'],x['negative'],len(x['rows']),d['status'],d['action'],round(d['kept_conf'],4),round(d['kept_frac'],4),d['n_kept'],';'.join(d['reasons'])])
if overlap_bad: raise RuntimeError(f'Nhãn road chồng lấn flood ở {len(overlap_bad)} ảnh: {overlap_bad[:10]}')


## Output

Dataset nằm tại `/kaggle/working/dataset` và preview tại `/kaggle/working/preview`. Download folder hoặc nén nó thành Kaggle Dataset, sau đó add làm input cho notebook (b). Test chỉ chứa group WaterDataset đã giữ riêng; không đưa ảnh test sang train/val.

File debug road: `/kaggle/working/road_debug.csv`, `/kaggle/working/preview_road_rejected`. Nên xem cả hai trước khi chỉnh ngưỡng trong cell cấu hình road.